# Loan Delinquency Prediction Model
## ML Pipeline Development

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET = "target_delinquent"
REFERENCE_YEAR = 2016
DATA_PATH = Path("loans.csv")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("default")

print(f"Data path: {DATA_PATH.resolve()}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")


Data path: /content/loans.csv
Artifacts directory: /content/artifacts


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 1: Load and Explore the Dataset

The original notebook had a good basic load-and-inspect section. We keep that idea, but make the path portable by
reading `loans.csv` from the project workspace instead of from a personal machine path.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected dataset at {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nColumn dtypes:")
display(df.dtypes.rename("dtype").to_frame())
print("\nBasic numeric summary:")
display(df.describe(include=[np.number]).T)


Dataset Shape: (10000, 40)

First 5 rows:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,emp_length,state,homeownership,annual_income,debt_to_income,annual_income_joint,debt_to_income_joint,delinq_2y,months_since_last_delinq,earliest_credit_line,inquiries_last_12m,total_credit_lines,open_credit_lines,total_credit_limit,total_credit_utilized,num_collections_last_12m,num_historical_failed_to_pay,months_since_90d_late,current_accounts_delinq,total_collection_amount_ever,current_installment_accounts,accounts_opened_24m,months_since_last_credit_inquiry,num_satisfactory_accounts,num_accounts_120d_past_due,num_accounts_30d_past_due,num_active_debit_accounts,total_debit_limit,num_total_cc_accounts,num_open_cc_accounts,num_cc_carrying_balance,num_mort_accounts,account_never_delinq_percent,tax_liens,public_record_bankrupt,loan_purpose,application_type,loan_amount,term,target_delinquent
0,3.0,NJ,MORTGAGE,90000.0,18.01,NaN,NaN,0,38.0,2001,6,28,10,70795,38767,0,0,38.0,0,1250,2,5,5.0,10,0.0,0,2,11100,14,8,6,1,92.9,0,0,moving,individual,28000,60,0
1,10.0,HI,RENT,40000.0,5.04,NaN,NaN,0,NaN,1996,1,30,14,28800,4321,0,1,NaN,0,0,0,11,8.0,14,0.0,0,3,16500,24,14,4,0,100.0,0,1,debt_consolidation,individual,5000,36,0
2,3.0,WI,RENT,40000.0,21.15,NaN,NaN,0,28.0,2006,4,31,10,24193,16000,0,0,28.0,0,432,1,13,7.0,10,0.0,0,3,4300,14,8,6,0,93.5,0,0,other,individual,2000,36,0
3,1.0,PA,RENT,30000.0,10.16,NaN,NaN,0,NaN,2007,0,4,4,25400,4997,0,1,NaN,0,0,1,1,15.0,4,0.0,0,2,19400,3,3,2,0,100.0,1,0,debt_consolidation,individual,21600,36,0
4,10.0,CA,RENT,35000.0,57.96,57000.0,37.66,0,NaN,2008,7,22,16,69839,52722,0,0,NaN,0,0,1,6,4.0,16,0.0,0,10,32700,20,15,13,0,100.0,0,0,credit_card,joint,23000,36,0



Column dtypes:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,dtype
emp_length,float64
state,object
homeownership,object
annual_income,float64
debt_to_income,float64
annual_income_joint,float64
debt_to_income_joint,float64
delinq_2y,int64
months_since_last_delinq,float64
earliest_credit_line,int64



Basic numeric summary:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count,mean,std,min,25%,50%,75%,max
emp_length,9183.0,5.930306,3.703734,0.00,2.0000,6.00,10.0000,10.00
annual_income,10000.0,79222.148412,64734.290492,0.00,45000.0000,65000.00,95000.0000,2300000.00
debt_to_income,9976.0,19.308192,15.004851,0.00,11.0575,17.57,25.0025,469.09
annual_income_joint,1495.0,127914.571244,70168.375404,19200.00,86833.5000,113000.00,151545.5000,1100000.00
debt_to_income_joint,1495.0,19.979304,8.054781,0.32,14.1600,19.72,25.5000,39.98
delinq_2y,10000.0,0.216000,0.683660,0.00,0.0000,0.00,0.0000,13.00
months_since_last_delinq,4342.0,36.760709,21.634939,1.00,19.0000,34.00,53.0000,118.00
earliest_credit_line,10000.0,2001.290000,7.795510,1963.00,1997.0000,2003.00,2006.0000,2015.00
inquiries_last_12m,10000.0,1.958200,2.380130,0.00,0.0000,1.00,3.0000,29.00
total_credit_lines,10000.0,22.679600,11.885439,2.00,14.0000,21.00,29.0000,87.00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 2: Validate Dataset Schema

This section mirrors the documented schema-validation stage: inspect types, missingness, and target balance before
making any modeling decisions.


In [ ]:
schema_report = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notnull().sum().values,
        "null_count": df.isnull().sum().values,
        "null_pct": (df.isnull().mean().values * 100).round(2),
    }
).sort_values(["null_pct", "column"], ascending=[False, True]).reset_index(drop=True)

numeric_cols_raw = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_raw = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("SCHEMA VALIDATION REPORT")
print("=" * 80)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Numeric columns: {len(numeric_cols_raw)}")
print(f"Categorical columns: {len(categorical_cols_raw)}")
print()
display(schema_report)

target_counts = df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count")
target_counts["pct"] = (target_counts["count"] / len(df) * 100).round(3)
print("Target distribution:")
display(target_counts)

missing_summary = schema_report.loc[schema_report["null_count"] > 0, ["column", "null_count", "null_pct"]]
print("Missing-value summary:")
display(missing_summary if not missing_summary.empty else pd.DataFrame({"message": ["No missing values found"]}))

duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows detected: {duplicate_rows}")


SCHEMA VALIDATION REPORT
Rows: 10,000
Columns: 40
Numeric columns: 36
Categorical columns: 4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,column,dtype,non_null_count,null_count,null_pct
0,annual_income_joint,float64,1495,8505,85.05
1,debt_to_income_joint,float64,1495,8505,85.05
2,months_since_90d_late,float64,2285,7715,77.15
3,months_since_last_delinq,float64,4342,5658,56.58
4,months_since_last_credit_inquiry,float64,8729,1271,12.71
5,emp_length,float64,9183,817,8.17
6,num_accounts_120d_past_due,float64,9682,318,3.18
7,debt_to_income,float64,9976,24,0.24
8,account_never_delinq_percent,float64,10000,0,0.00
9,accounts_opened_24m,int64,10000,0,0.00


Target distribution:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count,pct
target_delinquent,,
0,9822,98.22
1,178,1.78


Missing-value summary:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,column,null_count,null_pct
0,annual_income_joint,8505,85.05
1,debt_to_income_joint,8505,85.05
2,months_since_90d_late,7715,77.15
3,months_since_last_delinq,5658,56.58
4,months_since_last_credit_inquiry,1271,12.71
5,emp_length,817,8.17
6,num_accounts_120d_past_due,318,3.18
7,debt_to_income,24,0.24


Duplicate rows detected: 0


## Section 3: Clean the Dataset

The document's cleaning stage contains several domain-specific ideas that were missing or only partially implemented
in the original notebook. We keep the useful structure, but separate **structural cleaning** from **train-fitted
imputations** so we do not leak information from validation or test rows.

Structural cleaning done here:

- remove exact duplicates
- standardize target handling
- unify joint-income fields into application-level household measures
- encode event-never-happened semantics with sentinel values and explicit flags
- add a missingness flag for `emp_length`
- drop `num_accounts_120d_past_due` if it behaves like a near-constant reporting artifact


In [ ]:
structural_df = df.copy()
structural_df.columns = structural_df.columns.str.strip()

rows_before = len(structural_df)
duplicate_rows = structural_df.duplicated().sum()
if duplicate_rows:
    structural_df = structural_df.drop_duplicates().copy()

structural_df[TARGET] = pd.to_numeric(structural_df[TARGET], errors="coerce")
structural_df = structural_df.loc[structural_df[TARGET].notna()].copy()
structural_df[TARGET] = structural_df[TARGET].astype(int)

dropped_cols = []
if "num_accounts_120d_past_due" in structural_df.columns:
    non_null_unique = structural_df["num_accounts_120d_past_due"].dropna().nunique()
    if non_null_unique <= 1:
        structural_df = structural_df.drop(columns=["num_accounts_120d_past_due"])
        dropped_cols.append("num_accounts_120d_past_due")

if {"annual_income", "annual_income_joint", "application_type"}.issubset(structural_df.columns):
    joint_income_available = (
        structural_df["application_type"].astype(str).str.lower().eq("joint")
        & structural_df["annual_income_joint"].notna()
    )
    structural_df["combined_annual_income"] = np.where(
        joint_income_available,
        structural_df["annual_income_joint"],
        structural_df["annual_income"],
    )
    structural_df["used_joint_income"] = joint_income_available.astype(int)

if {"debt_to_income", "debt_to_income_joint", "application_type"}.issubset(structural_df.columns):
    joint_dti_available = (
        structural_df["application_type"].astype(str).str.lower().eq("joint")
        & structural_df["debt_to_income_joint"].notna()
    )
    structural_df["combined_dti"] = np.where(
        joint_dti_available,
        structural_df["debt_to_income_joint"],
        structural_df["debt_to_income"],
    )
    structural_df["used_joint_dti"] = joint_dti_available.astype(int)

for original_joint_col in ["annual_income_joint", "debt_to_income_joint"]:
    if original_joint_col in structural_df.columns:
        structural_df = structural_df.drop(columns=[original_joint_col])
        dropped_cols.append(original_joint_col)

event_cols = [
    "months_since_last_delinq",
    "months_since_90d_late",
    "months_since_last_credit_inquiry",
]
for col in event_cols:
    if col in structural_df.columns:
        never_col = f"{col}_never"
        structural_df[never_col] = structural_df[col].isna().astype(int)
        structural_df[col] = structural_df[col].fillna(999)

if "emp_length" in structural_df.columns:
    structural_df["emp_length_missing"] = structural_df["emp_length"].isna().astype(int)

# Keep the target in the final column for readability.
ordered_cols = [c for c in structural_df.columns if c != TARGET] + [TARGET]
structural_df = structural_df[ordered_cols]

print("STRUCTURAL CLEANING SUMMARY")
print("=" * 80)
print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {len(structural_df):,}")
print(f"Duplicates removed:   {duplicate_rows:,}")
print(f"Columns dropped:      {dropped_cols if dropped_cols else 'None'}")
print(f"Remaining missing values: {int(structural_df.isna().sum().sum()):,}")
display(structural_df.head())


STRUCTURAL CLEANING SUMMARY
Rows before cleaning: 10,000
Rows after cleaning:  10,000
Duplicates removed:   0
Columns dropped:      ['num_accounts_120d_past_due', 'annual_income_joint', 'debt_to_income_joint']
Remaining missing values: 841


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,emp_length,state,homeownership,annual_income,debt_to_income,delinq_2y,months_since_last_delinq,earliest_credit_line,inquiries_last_12m,total_credit_lines,open_credit_lines,total_credit_limit,total_credit_utilized,num_collections_last_12m,num_historical_failed_to_pay,months_since_90d_late,current_accounts_delinq,total_collection_amount_ever,current_installment_accounts,accounts_opened_24m,months_since_last_credit_inquiry,num_satisfactory_accounts,num_accounts_30d_past_due,num_active_debit_accounts,total_debit_limit,num_total_cc_accounts,num_open_cc_accounts,num_cc_carrying_balance,num_mort_accounts,account_never_delinq_percent,tax_liens,public_record_bankrupt,loan_purpose,application_type,loan_amount,term,combined_annual_income,used_joint_income,combined_dti,used_joint_dti,months_since_last_delinq_never,months_since_90d_late_never,months_since_last_credit_inquiry_never,emp_length_missing,target_delinquent
0,3.0,NJ,MORTGAGE,90000.0,18.01,0,38.0,2001,6,28,10,70795,38767,0,0,38.0,0,1250,2,5,5.0,10,0,2,11100,14,8,6,1,92.9,0,0,moving,individual,28000,60,90000.0,0,18.01,0,0,0,0,0,0
1,10.0,HI,RENT,40000.0,5.04,0,999.0,1996,1,30,14,28800,4321,0,1,999.0,0,0,0,11,8.0,14,0,3,16500,24,14,4,0,100.0,0,1,debt_consolidation,individual,5000,36,40000.0,0,5.04,0,1,1,0,0,0
2,3.0,WI,RENT,40000.0,21.15,0,28.0,2006,4,31,10,24193,16000,0,0,28.0,0,432,1,13,7.0,10,0,3,4300,14,8,6,0,93.5,0,0,other,individual,2000,36,40000.0,0,21.15,0,0,0,0,0,0
3,1.0,PA,RENT,30000.0,10.16,0,999.0,2007,0,4,4,25400,4997,0,1,999.0,0,0,1,1,15.0,4,0,2,19400,3,3,2,0,100.0,1,0,debt_consolidation,individual,21600,36,30000.0,0,10.16,0,1,1,0,0,0
4,10.0,CA,RENT,35000.0,57.96,0,999.0,2008,7,22,16,69839,52722,0,0,999.0,0,0,1,6,4.0,16,0,10,32700,20,15,13,0,100.0,0,0,credit_card,joint,23000,36,57000.0,1,37.66,1,1,1,0,0,0


## Section 4: Feature Engineering and Preprocessing

This notebook now follows the document's feature-engineering trajectory more closely:

- affordability features
- utilization features
- credit-velocity features
- aggregate delinquency burden
- geography bucketing (`state` -> `region`)
- log transforms for skewed monetary variables

We also intentionally demonstrate the **broken** debit-utilization idea from the document, then drop it.


In [ ]:
feature_df = structural_df.copy()

def safe_divide(numerator, denominator, fill_value=0.0):
    numerator = pd.Series(numerator, index=feature_df.index, dtype=float)
    denominator = pd.Series(denominator, index=feature_df.index, dtype=float).replace(0, np.nan)
    values = numerator.div(denominator)
    return values.replace([np.inf, -np.inf], np.nan).fillna(fill_value)

if "earliest_credit_line" in feature_df.columns:
    feature_df["earliest_credit_line"] = pd.to_numeric(feature_df["earliest_credit_line"], errors="coerce")
    feature_df["credit_history_years"] = (REFERENCE_YEAR - feature_df["earliest_credit_line"]).clip(lower=0)
    feature_df = feature_df.drop(columns=["earliest_credit_line"])

feature_df["loan_to_income"] = safe_divide(feature_df["loan_amount"], feature_df["combined_annual_income"])
feature_df["installment_to_income"] = safe_divide(
    feature_df["loan_amount"] / feature_df["term"].replace(0, np.nan),
    feature_df["combined_annual_income"] / 12,
)
feature_df["credit_utilization_ratio"] = safe_divide(
    feature_df["total_credit_utilized"], feature_df["total_credit_limit"]
)
feature_df["open_credit_ratio"] = safe_divide(
    feature_df["open_credit_lines"], feature_df["total_credit_lines"]
)
feature_df["cc_balance_ratio"] = safe_divide(
    feature_df["num_cc_carrying_balance"], feature_df["num_open_cc_accounts"]
)
feature_df["open_cc_ratio"] = safe_divide(
    feature_df["num_open_cc_accounts"], feature_df["num_total_cc_accounts"]
)
feature_df["recent_account_ratio"] = safe_divide(
    feature_df["accounts_opened_24m"], feature_df["total_credit_lines"]
)
feature_df["accounts_per_credit_year"] = safe_divide(
    feature_df["total_credit_lines"], feature_df["credit_history_years"].replace(0, np.nan)
)
feature_df["inquiries_per_credit_year"] = safe_divide(
    feature_df["inquiries_last_12m"], feature_df["credit_history_years"].replace(0, np.nan)
)
feature_df["total_negative_marks"] = (
    feature_df["delinq_2y"].fillna(0)
    + feature_df["num_collections_last_12m"].fillna(0)
    + feature_df["num_historical_failed_to_pay"].fillna(0)
    + feature_df["tax_liens"].fillna(0)
    + feature_df["public_record_bankrupt"].fillna(0)
    + feature_df["current_accounts_delinq"].fillna(0)
    + feature_df["num_accounts_30d_past_due"].fillna(0)
)
feature_df["has_delinq_history"] = (feature_df["total_negative_marks"] > 0).astype(int)
feature_df["log_combined_annual_income"] = np.log1p(feature_df["combined_annual_income"].clip(lower=0))
feature_df["log_loan_amount"] = np.log1p(feature_df["loan_amount"].clip(lower=0))

state_to_region = {
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast", "NH": "Northeast",
    "RI": "Northeast", "VT": "Northeast", "NJ": "Northeast", "NY": "Northeast", "PA": "Northeast",
    "IL": "Midwest", "IN": "Midwest", "MI": "Midwest", "OH": "Midwest", "WI": "Midwest",
    "IA": "Midwest", "KS": "Midwest", "MN": "Midwest", "MO": "Midwest", "NE": "Midwest",
    "ND": "Midwest", "SD": "Midwest",
    "DE": "South", "FL": "South", "GA": "South", "MD": "South", "NC": "South",
    "SC": "South", "VA": "South", "DC": "South", "WV": "South", "AL": "South",
    "KY": "South", "MS": "South", "TN": "South", "AR": "South", "LA": "South",
    "OK": "South", "TX": "South",
    "AZ": "West", "CO": "West", "ID": "West", "MT": "West", "NV": "West",
    "NM": "West", "UT": "West", "WY": "West", "AK": "West", "CA": "West",
    "HI": "West", "OR": "West", "WA": "West",
}
if "state" in feature_df.columns:
    feature_df["region"] = feature_df["state"].map(state_to_region).fillna("Unknown")

# Diagnostic only: this intentionally mirrors the document's "bug moment".
buggy_debit_utilization = safe_divide(feature_df["total_credit_utilized"], feature_df["total_debit_limit"])
print("Diagnostic for intentionally rejected feature `debit_utilization_ratio`:")
display(buggy_debit_utilization.describe(percentiles=[0.5, 0.9, 0.99]).to_frame("value").T)
print("The ratio is not kept because it mixes credit utilization with debit limits, which is semantically inconsistent.")

engineered_cols = [
    "combined_annual_income",
    "combined_dti",
    "loan_to_income",
    "installment_to_income",
    "credit_utilization_ratio",
    "open_credit_ratio",
    "cc_balance_ratio",
    "open_cc_ratio",
    "recent_account_ratio",
    "accounts_per_credit_year",
    "inquiries_per_credit_year",
    "total_negative_marks",
    "has_delinq_history",
    "log_combined_annual_income",
    "log_loan_amount",
]
print("\nEngineered feature summary:")
display(feature_df[engineered_cols].describe().T[["mean", "std", "min", "max"]].round(4))


Diagnostic for intentionally rejected feature `debit_utilization_ratio`:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count,mean,std,min,50%,90%,99%,max
value,10000.0,3.699822,8.732259,0.0,1.690776,7.475986,33.804633,247.936667


The ratio is not kept because it mixes credit utilization with debit limits, which is semantically inconsistent.

Engineered feature summary:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,mean,std,min,max
combined_annual_income,89138.0193,69463.6793,5235.0000,2.300000e+06
combined_dti,17.7341,8.7061,0.0000,3.999000e+01
loan_to_income,0.2153,0.1344,0.0013,1.068300e+00
installment_to_income,0.0607,0.0382,0.0004,3.356000e-01
credit_utilization_ratio,0.4031,0.2693,0.0000,1.835300e+00
open_credit_ratio,0.5429,0.1844,0.0000,1.000000e+00
cc_balance_ratio,0.6792,0.2419,0.0000,2.000000e+00
open_cc_ratio,0.6604,0.2077,0.0000,1.000000e+00
recent_account_ratio,0.2169,0.1567,0.0000,1.000000e+00
accounts_per_credit_year,1.8874,1.2598,0.1034,1.266670e+01


## Section 5: Data Split

The original notebook already had the right instinct here: create stratified train/validation/test splits and fit
downstream transformations on the training data only. We keep that structure and make the split explicit:

- `70%` train
- `10%` validation
- `20%` test


In [ ]:
train_df, temp_df = train_test_split(
    feature_df,
    test_size=0.30,
    stratify=feature_df[TARGET],
    random_state=RANDOM_STATE,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=(2 / 3),
    stratify=temp_df[TARGET],
    random_state=RANDOM_STATE,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Split shapes:")
print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")

split_distribution = []
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_distribution.append(
        {
            "split": split_name,
            "rows": len(split_df),
            "positive_count": int(split_df[TARGET].sum()),
            "positive_rate_pct": round(split_df[TARGET].mean() * 100, 3),
        }
    )
display(pd.DataFrame(split_distribution))


Split shapes:
Train: (7000, 59)
Val:   (1000, 59)
Test:  (2000, 59)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,split,rows,positive_count,positive_rate_pct
0,train,7000,125,1.786
1,val,1000,18,1.800
2,test,2000,35,1.750


## Section 5b: Leakage-Safe Train-Fitted Imputation

This section is new compared with the original notebook, and it closes the biggest methodological gap:
any imputation that learns a median or mode must be fit on `train` only.

We also simplify the modeling space slightly by using `region` as the geographic feature and dropping raw `state`
from the encoded design matrix. That keeps the notebook closer to the documented "lower-cardinality geography"
strategy while remaining easy to follow.


In [ ]:
drop_from_model = []
if "state" in train_df.columns and "region" in train_df.columns:
    drop_from_model.append("state")

train_model_df = train_df.drop(columns=drop_from_model).copy()
val_model_df = val_df.drop(columns=drop_from_model).copy()
test_model_df = test_df.drop(columns=drop_from_model).copy()

feature_cols = [c for c in train_model_df.columns if c != TARGET]
numeric_cols = train_model_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

train_numeric = pd.DataFrame(
    numeric_imputer.fit_transform(train_model_df[numeric_cols]),
    columns=numeric_cols,
    index=train_model_df.index,
)
val_numeric = pd.DataFrame(
    numeric_imputer.transform(val_model_df[numeric_cols]),
    columns=numeric_cols,
    index=val_model_df.index,
)
test_numeric = pd.DataFrame(
    numeric_imputer.transform(test_model_df[numeric_cols]),
    columns=numeric_cols,
    index=test_model_df.index,
)

if categorical_cols:
    train_categorical = pd.DataFrame(
        categorical_imputer.fit_transform(train_model_df[categorical_cols]),
        columns=categorical_cols,
        index=train_model_df.index,
    )
    val_categorical = pd.DataFrame(
        categorical_imputer.transform(val_model_df[categorical_cols]),
        columns=categorical_cols,
        index=val_model_df.index,
    )
    test_categorical = pd.DataFrame(
        categorical_imputer.transform(test_model_df[categorical_cols]),
        columns=categorical_cols,
        index=test_model_df.index,
    )
else:
    train_categorical = pd.DataFrame(index=train_model_df.index)
    val_categorical = pd.DataFrame(index=val_model_df.index)
    test_categorical = pd.DataFrame(index=test_model_df.index)

y_train = train_model_df[TARGET].astype(int).copy()
y_val = val_model_df[TARGET].astype(int).copy()
y_test = test_model_df[TARGET].astype(int).copy()

print("Imputation fitted on train only.")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Columns dropped from modeling view: {drop_from_model if drop_from_model else 'None'}")


Imputation fitted on train only.
Numeric columns: 53
Categorical columns: 4
Columns dropped from modeling view: ['state']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 6: Scale / Normalize Features

The original notebook's training-only scaling idea was good, so we retain it. We scale continuous numeric features,
leave binary flags unchanged, and one-hot encode the categorical columns using the training categories as the source
of truth.


In [ ]:
def is_binary_like(series):
    values = pd.Series(series).dropna().unique().tolist()
    return set(values).issubset({0, 1})

binary_cols = [col for col in numeric_cols if is_binary_like(train_numeric[col])]
continuous_cols = [col for col in numeric_cols if col not in binary_cols]

scaler = StandardScaler()
train_numeric_scaled = train_numeric.copy()
val_numeric_scaled = val_numeric.copy()
test_numeric_scaled = test_numeric.copy()

if continuous_cols:
    train_numeric_scaled[continuous_cols] = scaler.fit_transform(train_numeric[continuous_cols])
    val_numeric_scaled[continuous_cols] = scaler.transform(val_numeric[continuous_cols])
    test_numeric_scaled[continuous_cols] = scaler.transform(test_numeric[continuous_cols])

for col in categorical_cols:
    train_categorical[col] = pd.Categorical(train_categorical[col])
    val_categorical[col] = pd.Categorical(val_categorical[col], categories=train_categorical[col].cat.categories)
    test_categorical[col] = pd.Categorical(test_categorical[col], categories=train_categorical[col].cat.categories)

train_encoded = pd.get_dummies(train_categorical, dummy_na=False)
val_encoded = pd.get_dummies(val_categorical, dummy_na=False).reindex(columns=train_encoded.columns, fill_value=0)
test_encoded = pd.get_dummies(test_categorical, dummy_na=False).reindex(columns=train_encoded.columns, fill_value=0)

X_train = pd.concat([train_numeric_scaled, train_encoded], axis=1)
X_val = pd.concat([val_numeric_scaled, val_encoded], axis=1).reindex(columns=X_train.columns, fill_value=0)
X_test = pd.concat([test_numeric_scaled, test_encoded], axis=1).reindex(columns=X_train.columns, fill_value=0)

preprocess_artifact = {
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "binary_columns": binary_cols,
    "continuous_columns": continuous_cols,
    "feature_columns": X_train.columns.tolist(),
    "dropped_raw_columns": drop_from_model,
    "reference_year": REFERENCE_YEAR,
    "numeric_imputer": numeric_imputer,
    "categorical_imputer": categorical_imputer,
    "scaler": scaler,
}
joblib.dump(preprocess_artifact, ARTIFACTS_DIR / "preprocess_bundle.joblib")

print(f"Scaled continuous columns: {len(continuous_cols)}")
print(f"Binary columns left unscaled: {len(binary_cols)}")
print(f"Encoded categorical features: {train_encoded.shape[1]}")
print(f"Final train matrix shape: {X_train.shape}")


Scaled continuous columns: 44
Binary columns left unscaled: 9
Encoded categorical features: 21
Final train matrix shape: (7000, 74)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 7: Baseline Models

We keep the original notebook's baseline-model spirit, but we restrict model selection to the validation set only.
The test set stays untouched until the final stage.


In [ ]:
def probs_from_model(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    preds = model.predict(X)
    return np.asarray(preds, dtype=float)


def best_f1_threshold(y_true, probs):
    candidate_thresholds = np.unique(
        np.clip(
            np.quantile(probs, np.linspace(0.05, 0.95, 91)),
            0.001,
            0.999,
        )
    )
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in candidate_thresholds:
        preds = (probs >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def metrics_from_probs(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        "roc_auc": roc_auc_score(y_true, probs),
        "pr_auc": average_precision_score(y_true, probs),
        "brier": brier_score_loss(y_true, probs),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


pos = int(y_train.sum())
neg = int(len(y_train) - pos)
scale_pos_weight = neg / pos if pos else 1.0
print(f"Training positives: {pos}")
print(f"Training negatives: {neg}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

baseline_specs = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    ),
}

optional_model_errors = {}
try:
    from xgboost import XGBClassifier

    baseline_specs["XGBoost"] = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
    )
except Exception as exc:
    optional_model_errors["XGBoost"] = str(exc)

try:
    from lightgbm import LGBMClassifier

    baseline_specs["LightGBM"] = LGBMClassifier(
        objective="binary",
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
    )
except Exception as exc:
    optional_model_errors["LightGBM"] = str(exc)

baseline_models = {}
baseline_rows = []
for model_name, estimator in baseline_specs.items():
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)
    val_probs = probs_from_model(fitted, X_val)
    best_thr, best_f1 = best_f1_threshold(y_val, val_probs)
    val_metrics = metrics_from_probs(y_val, val_probs, best_thr)
    baseline_models[model_name] = {
        "model": fitted,
        "val_probs": val_probs,
        "threshold": best_thr,
    }
    baseline_rows.append(
        {
            "model": model_name,
            "stage": "baseline",
            "val_threshold": round(best_thr, 4),
            "val_f1_best": round(best_f1, 4),
            "val_roc_auc": round(val_metrics["roc_auc"], 4),
            "val_pr_auc": round(val_metrics["pr_auc"], 4),
            "val_brier": round(val_metrics["brier"], 4),
            "val_tp": val_metrics["tp"],
            "val_fp": val_metrics["fp"],
        }
    )

baseline_results = pd.DataFrame(baseline_rows).sort_values(
    ["val_pr_auc", "val_roc_auc", "val_f1_best"],
    ascending=False,
).reset_index(drop=True)
display(baseline_results)

if optional_model_errors:
    print("Optional model imports unavailable:")
    display(pd.DataFrame({"model": list(optional_model_errors), "reason": list(optional_model_errors.values())}))


Training positives: 125
Training negatives: 6875
scale_pos_weight: 55.00
[LightGBM] [Info] Number of positive: 125, number of negative: 6875
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002851 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5600
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 71
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.017857 -> initscore=-4.007333
[LightGBM] [Info] Start training from score -4.007333


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,model,stage,val_threshold,val_f1_best,val_roc_auc,val_pr_auc,val_brier,val_tp,val_fp
0,HistGradientBoosting,baseline,0.0012,0.0370,0.6518,0.0277,0.0180,2,88
1,Logistic Regression,baseline,0.8012,0.0588,0.5292,0.0258,0.2143,2,48
2,XGBoost,baseline,0.1726,0.0595,0.5609,0.0223,0.0393,5,145
3,LightGBM,baseline,0.0022,0.0403,0.5606,0.0222,0.0186,5,225
4,Random Forest,baseline,0.1022,0.0612,0.5303,0.0220,0.0203,3,77


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 8: Hyperparameter Tuning

The original notebook attempted tuning but failed because of an incompatible `early_stopping_rounds` usage.
This version keeps tuning simpler and more robust:

- cross-validated search on the training split only
- no validation-set leakage inside the CV search
- validation split used afterward only for model comparison and threshold selection


In [ ]:
tuned_candidates = {}
tuned_rows = []

def fit_and_score_candidate(model_name, estimator, param_distributions, n_iter=12):
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="average_precision",
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        error_score="raise",
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_model.fit(X_train, y_train)
    val_probs = probs_from_model(best_model, X_val)
    best_thr, best_f1 = best_f1_threshold(y_val, val_probs)
    val_metrics = metrics_from_probs(y_val, val_probs, best_thr)
    tuned_candidates[model_name] = {
        "model": best_model,
        "val_probs": val_probs,
        "threshold": best_thr,
        "cv_best_score": float(search.best_score_),
        "best_params": search.best_params_,
    }
    tuned_rows.append(
        {
            "model": model_name,
            "stage": "tuned",
            "cv_pr_auc": round(float(search.best_score_), 4),
            "val_threshold": round(best_thr, 4),
            "val_f1_best": round(best_f1, 4),
            "val_roc_auc": round(val_metrics["roc_auc"], 4),
            "val_pr_auc": round(val_metrics["pr_auc"], 4),
            "val_brier": round(val_metrics["brier"], 4),
        }
    )


tuning_errors = {}

try:
    fit_and_score_candidate(
        "HistGradientBoosting (tuned)",
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "learning_rate": [0.03, 0.05, 0.08, 0.1],
            "max_iter": [150, 250, 350],
            "max_leaf_nodes": [15, 31, 63],
            "min_samples_leaf": [20, 40, 80],
            "l2_regularization": [0.0, 0.1, 1.0],
        },
        n_iter=12,
    )
except Exception as exc:
    tuning_errors["HistGradientBoosting (tuned)"] = str(exc)

if "XGBoost" in baseline_specs:
    try:
        fit_and_score_candidate(
            "XGBoost (tuned)",
            baseline_specs["XGBoost"].set_params(use_label_encoder=False if "use_label_encoder" in baseline_specs["XGBoost"].get_params() else None),
            {
                "n_estimators": [150, 250, 350, 500],
                "max_depth": [3, 4, 5, 6],
                "learning_rate": [0.03, 0.05, 0.08, 0.1],
                "subsample": [0.7, 0.8, 0.9, 1.0],
                "colsample_bytree": [0.6, 0.8, 1.0],
                "min_child_weight": [1, 3, 5],
                "gamma": [0.0, 0.5, 1.0],
            },
            n_iter=15,
        )
    except Exception as exc:
        tuning_errors["XGBoost (tuned)"] = str(exc)

if "LightGBM" in baseline_specs:
    try:
        fit_and_score_candidate(
            "LightGBM (tuned)",
            baseline_specs["LightGBM"],
            {
                "n_estimators": [150, 250, 350, 500],
                "learning_rate": [0.03, 0.05, 0.08, 0.1],
                "num_leaves": [15, 31, 63],
                "subsample": [0.7, 0.8, 0.9, 1.0],
                "colsample_bytree": [0.6, 0.8, 1.0],
                "min_child_samples": [10, 20, 40],
            },
            n_iter=12,
        )
    except Exception as exc:
        tuning_errors["LightGBM (tuned)"] = str(exc)

tuned_results = pd.DataFrame(tuned_rows).sort_values(
    ["val_pr_auc", "val_roc_auc", "val_f1_best"],
    ascending=False,
).reset_index(drop=True)
display(tuned_results if not tuned_results.empty else pd.DataFrame({"message": ["No tuned models completed"]}))

if tuning_errors:
    print("Tuning errors:")
    display(pd.DataFrame({"model": list(tuning_errors), "reason": list(tuning_errors.values())}))

combined_results = pd.concat([baseline_results, tuned_results], ignore_index=True, sort=False).sort_values(
    ["val_pr_auc", "val_roc_auc", "val_f1_best"],
    ascending=False,
).reset_index(drop=True)
display(combined_results)


[LightGBM] [Info] Number of positive: 125, number of negative: 6875
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002485 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5600
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 71
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.017857 -> initscore=-4.007333
[LightGBM] [Info] Start training from score -4.007333


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[LightGBM] [Info] Number of positive: 125, number of negative: 6875
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002494 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5600
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 71
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.017857 -> initscore=-4.007333
[LightGBM] [Info] Start training from score -4.007333
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,model,stage,cv_pr_auc,val_threshold,val_f1_best,val_roc_auc,val_pr_auc,val_brier
0,LightGBM (tuned),tuned,0.0386,0.0010,0.0000,0.6194,0.0248,0.0180
1,HistGradientBoosting (tuned),tuned,0.0434,0.0010,0.0455,0.5652,0.0227,0.0180
2,XGBoost (tuned),tuned,0.0457,0.0347,0.0513,0.5260,0.0212,0.0212


,model,stage,val_threshold,val_f1_best,val_roc_auc,val_pr_auc,val_brier,val_tp,val_fp,cv_pr_auc
0,HistGradientBoosting,baseline,0.0012,0.0370,0.6518,0.0277,0.0180,2.0,88.0,NaN
1,Logistic Regression,baseline,0.8012,0.0588,0.5292,0.0258,0.2143,2.0,48.0,NaN
2,LightGBM (tuned),tuned,0.0010,0.0000,0.6194,0.0248,0.0180,NaN,NaN,0.0386
3,HistGradientBoosting (tuned),tuned,0.0010,0.0455,0.5652,0.0227,0.0180,NaN,NaN,0.0434
4,XGBoost,baseline,0.1726,0.0595,0.5609,0.0223,0.0393,5.0,145.0,NaN
5,LightGBM,baseline,0.0022,0.0403,0.5606,0.0222,0.0186,5.0,225.0,NaN
6,Random Forest,baseline,0.1022,0.0612,0.5303,0.0220,0.0203,3.0,77.0,NaN
7,XGBoost (tuned),tuned,0.0347,0.0513,0.5260,0.0212,0.0212,NaN,NaN,0.0457


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 9: Feature Importance and Pruning Check

The document includes an important teaching moment here: pruning features with zero gain does not always improve
generalization. We reproduce that logic as a controlled experiment on the validation set only.


In [ ]:
candidate_registry = {}
candidate_registry.update(baseline_models)
candidate_registry.update(tuned_candidates)

champion_name = combined_results.iloc[0]["model"]
champion_bundle = candidate_registry[champion_name]
champion_model = champion_bundle["model"]
champion_threshold = champion_bundle["threshold"]
champion_val_probs = champion_bundle["val_probs"]
champion_val_metrics = metrics_from_probs(y_val, champion_val_probs, champion_threshold)

selected_features = X_train.columns.tolist()
pruning_summary = None

if hasattr(champion_model, "feature_importances_"):
    importance_df = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": champion_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    zero_gain_features = importance_df.loc[importance_df["importance"] <= 0, "feature"].tolist()
    display(importance_df.head(20))

    if 0 < len(zero_gain_features) < X_train.shape[1]:
        retained_features = [f for f in X_train.columns if f not in zero_gain_features]
        pruned_model = clone(champion_model)
        pruned_model.fit(X_train[retained_features], y_train)
        pruned_val_probs = probs_from_model(pruned_model, X_val[retained_features])
        pruned_threshold, pruned_best_f1 = best_f1_threshold(y_val, pruned_val_probs)
        pruned_val_metrics = metrics_from_probs(y_val, pruned_val_probs, pruned_threshold)

        pruning_summary = pd.DataFrame(
            [
                {
                    "version": "full",
                    "feature_count": X_train.shape[1],
                    "val_pr_auc": round(champion_val_metrics["pr_auc"], 4),
                    "val_roc_auc": round(champion_val_metrics["roc_auc"], 4),
                    "val_f1_best": round(champion_val_metrics["f1"], 4),
                },
                {
                    "version": "pruned",
                    "feature_count": len(retained_features),
                    "val_pr_auc": round(pruned_val_metrics["pr_auc"], 4),
                    "val_roc_auc": round(pruned_val_metrics["roc_auc"], 4),
                    "val_f1_best": round(pruned_val_metrics["f1"], 4),
                },
            ]
        )
        display(pruning_summary)

        if pruned_val_metrics["pr_auc"] >= champion_val_metrics["pr_auc"] + 0.001:
            champion_name = f"{champion_name} + pruning"
            champion_model = pruned_model
            champion_threshold = pruned_threshold
            champion_val_probs = pruned_val_probs
            selected_features = retained_features
            champion_val_metrics = pruned_val_metrics
            print("Pruned version selected.")
        else:
            print("Full feature set retained because pruning did not produce a clear validation gain.")
    else:
        print("No meaningful zero-gain pruning opportunity detected.")
else:
    print(f"{champion_name} does not expose tree-style feature importances; pruning check skipped.")

X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()
X_test_selected = X_test[selected_features].copy()
champion_model.fit(X_train_selected, y_train)
champion_val_probs = probs_from_model(champion_model, X_val_selected)
champion_threshold, _ = best_f1_threshold(y_val, champion_val_probs)
champion_val_metrics = metrics_from_probs(y_val, champion_val_probs, champion_threshold)

print("Champion after pruning check:")
print(champion_name)
print(champion_val_metrics)


HistGradientBoosting does not expose tree-style feature importances; pruning check skipped.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Champion after pruning check:
HistGradientBoosting
{'roc_auc': np.float64(0.6517877347816248), 'pr_auc': np.float64(0.02768900959453606), 'brier': np.float64(0.018011833758742914), 'precision': 0.022222222222222223, 'recall': 0.1111111111111111, 'f1': 0.037037037037037035, 'tp': 2, 'fp': 88, 'fn': 16, 'tn': 894}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 10: Probability Calibration

The documentation emphasizes that ranking quality and probability quality are different goals. This section compares:

- uncalibrated champion model
- isotonic calibration
- sigmoid / Platt calibration

Calibration is fit using the validation predictions, and the final test set remains untouched until the next section.


In [ ]:
calibration_rows = []
calibration_artifacts = {}

uncal_val_probs = champion_val_probs.copy()

uncal_thr, _ = best_f1_threshold(y_val, uncal_val_probs)
uncal_metrics = metrics_from_probs(y_val, uncal_val_probs, uncal_thr)
calibration_rows.append(
    {
        "model": "uncalibrated",
        "val_threshold": round(uncal_thr, 4),
        "val_roc_auc": round(uncal_metrics["roc_auc"], 4),
        "val_pr_auc": round(uncal_metrics["pr_auc"], 4),
        "val_brier": round(uncal_metrics["brier"], 4),
        "val_f1_best": round(uncal_metrics["f1"], 4),
    }
)
calibration_artifacts["uncalibrated"] = {
    "val_probs": uncal_val_probs,
    "threshold": uncal_thr,
    "calibrator": None,
}

isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(uncal_val_probs, y_val)
isotonic_val_probs = isotonic.predict(uncal_val_probs)
isotonic_thr, _ = best_f1_threshold(y_val, isotonic_val_probs)
isotonic_metrics = metrics_from_probs(y_val, isotonic_val_probs, isotonic_thr)
calibration_rows.append(
    {
        "model": "isotonic",
        "val_threshold": round(isotonic_thr, 4),
        "val_roc_auc": round(isotonic_metrics["roc_auc"], 4),
        "val_pr_auc": round(isotonic_metrics["pr_auc"], 4),
        "val_brier": round(isotonic_metrics["brier"], 4),
        "val_f1_best": round(isotonic_metrics["f1"], 4),
    }
)
calibration_artifacts["isotonic"] = {
    "val_probs": isotonic_val_probs,
    "threshold": isotonic_thr,
    "calibrator": isotonic,
}

sigmoid = LogisticRegression(solver="lbfgs")
sigmoid.fit(uncal_val_probs.reshape(-1, 1), y_val)
sigmoid_val_probs = sigmoid.predict_proba(uncal_val_probs.reshape(-1, 1))[:, 1]
sigmoid_thr, _ = best_f1_threshold(y_val, sigmoid_val_probs)
sigmoid_metrics = metrics_from_probs(y_val, sigmoid_val_probs, sigmoid_thr)
calibration_rows.append(
    {
        "model": "sigmoid_platt",
        "val_threshold": round(sigmoid_thr, 4),
        "val_roc_auc": round(sigmoid_metrics["roc_auc"], 4),
        "val_pr_auc": round(sigmoid_metrics["pr_auc"], 4),
        "val_brier": round(sigmoid_metrics["brier"], 4),
        "val_f1_best": round(sigmoid_metrics["f1"], 4),
    }
)
calibration_artifacts["sigmoid_platt"] = {
    "val_probs": sigmoid_val_probs,
    "threshold": sigmoid_thr,
    "calibrator": sigmoid,
}

calibration_results = pd.DataFrame(calibration_rows).sort_values(
    ["val_brier", "val_pr_auc", "val_roc_auc"],
    ascending=[True, False, False],
).reset_index(drop=True)
display(calibration_results)

chosen_calibration = calibration_results.iloc[0]["model"]
val_positive_count = int(y_val.sum())
if val_positive_count < 50:
    isotonic_row = calibration_results.loc[calibration_results["model"] == "isotonic"]
    sigmoid_row = calibration_results.loc[calibration_results["model"] == "sigmoid_platt"]
    if not isotonic_row.empty and not sigmoid_row.empty:
        isotonic_brier = float(isotonic_row["val_brier"].iloc[0])
        sigmoid_brier = float(sigmoid_row["val_brier"].iloc[0])
        isotonic_pr = float(isotonic_row["val_pr_auc"].iloc[0])
        sigmoid_pr = float(sigmoid_row["val_pr_auc"].iloc[0])
        if sigmoid_brier <= isotonic_brier + 0.001 and sigmoid_pr >= isotonic_pr:
            chosen_calibration = "sigmoid_platt"

print(f"Validation positives: {val_positive_count}")
print(f"Chosen calibration: {chosen_calibration}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,model,val_threshold,val_roc_auc,val_pr_auc,val_brier,val_f1_best
0,isotonic,0.0327,0.7046,0.0304,0.0175,0.0621
1,sigmoid_platt,0.0180,0.3482,0.0133,0.0177,0.0362
2,uncalibrated,0.0012,0.6518,0.0277,0.0180,0.0370


Validation positives: 18
Chosen calibration: isotonic


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 11: Final Test Evaluation

This is the first place where the held-out test set is used. We compare:

- the uncalibrated champion model
- the chosen calibrated version

Thresholds come from the validation set and are applied unchanged to the test set.


In [ ]:
uncal_test_probs = probs_from_model(champion_model, X_test_selected)
uncal_test_metrics = metrics_from_probs(
    y_test,
    uncal_test_probs,
    calibration_artifacts["uncalibrated"]["threshold"],
)

final_test_rows = [
    {
        "model": "uncalibrated",
        "test_threshold": round(calibration_artifacts["uncalibrated"]["threshold"], 4),
        "test_roc_auc": round(uncal_test_metrics["roc_auc"], 4),
        "test_pr_auc": round(uncal_test_metrics["pr_auc"], 4),
        "test_brier": round(uncal_test_metrics["brier"], 4),
        "test_f1": round(uncal_test_metrics["f1"], 4),
        "tp": uncal_test_metrics["tp"],
        "fp": uncal_test_metrics["fp"],
        "fn": uncal_test_metrics["fn"],
    }
]

if chosen_calibration == "isotonic":
    calibrated_test_probs = calibration_artifacts["isotonic"]["calibrator"].predict(uncal_test_probs)
elif chosen_calibration == "sigmoid_platt":
    calibrated_test_probs = calibration_artifacts["sigmoid_platt"]["calibrator"].predict_proba(
        uncal_test_probs.reshape(-1, 1)
    )[:, 1]
else:
    calibrated_test_probs = uncal_test_probs.copy()

calibrated_test_metrics = metrics_from_probs(
    y_test,
    calibrated_test_probs,
    calibration_artifacts[chosen_calibration]["threshold"],
)
final_test_rows.append(
    {
        "model": chosen_calibration,
        "test_threshold": round(calibration_artifacts[chosen_calibration]["threshold"], 4),
        "test_roc_auc": round(calibrated_test_metrics["roc_auc"], 4),
        "test_pr_auc": round(calibrated_test_metrics["pr_auc"], 4),
        "test_brier": round(calibrated_test_metrics["brier"], 4),
        "test_f1": round(calibrated_test_metrics["f1"], 4),
        "tp": calibrated_test_metrics["tp"],
        "fp": calibrated_test_metrics["fp"],
        "fn": calibrated_test_metrics["fn"],
    }
)

final_test_results = pd.DataFrame(final_test_rows)
display(final_test_results)

prob_curves = []
for curve_name, probs in [("uncalibrated", uncal_test_probs), (chosen_calibration, calibrated_test_probs)]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy="quantile")
    prob_curves.append(
        pd.DataFrame(
            {
                "model": curve_name,
                "predicted_probability": mean_pred,
                "observed_rate": frac_pos,
            }
        )
    )
calibration_curve_df = pd.concat(prob_curves, ignore_index=True)
display(calibration_curve_df)


,model,test_threshold,test_roc_auc,test_pr_auc,test_brier,test_f1,tp,fp,fn
0,uncalibrated,0.0012,0.5903,0.0491,0.0174,0.0766,8,166,27
1,isotonic,0.0327,0.5746,0.0216,0.0173,0.0474,16,624,19


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,model,predicted_probability,observed_rate
0,uncalibrated,0.000007,0.010000
1,uncalibrated,0.000016,0.005000
2,uncalibrated,0.000027,0.030000
3,uncalibrated,0.000043,0.005000
4,uncalibrated,0.000067,0.025000
5,uncalibrated,0.000102,0.010000
6,uncalibrated,0.000163,0.015000
7,uncalibrated,0.000279,0.025000
8,uncalibrated,0.000591,0.010000
9,uncalibrated,0.003837,0.040000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 12: Final Recommendation and Human-Level Insight

This final section turns the metrics into an honest interpretation:

- if ranking metrics beat calibration metrics, the model is stronger as a ranking tool
- if Brier improves materially after calibration, calibrated probabilities are more trustworthy
- if validation performance drops sharply on test, that is a real generalization warning, not a notebook failure


In [ ]:
final_choice_row = final_test_results.loc[final_test_results["model"] == chosen_calibration].iloc[0]
ranking_best_row = final_test_results.sort_values(["test_pr_auc", "test_roc_auc"], ascending=False).iloc[0]
calibration_best_row = final_test_results.sort_values("test_brier", ascending=True).iloc[0]

In [ ]:
recommendation_lines = [
    f"Champion model before calibration: {champion_name}",
    f"Chosen calibration strategy: {chosen_calibration}",
    f"Best ranking profile on test: {ranking_best_row['model']}",
    f"Best probability calibration on test (lowest Brier): {calibration_best_row['model']}",
]


In [ ]:
if calibration_best_row["model"] != ranking_best_row["model"]:
    recommendation_lines.append(
        "Interpretation: this workflow likely produces one model that ranks risk best and another that emits more trustworthy probabilities."
    )
else:
    recommendation_lines.append(
        "Interpretation: the same final model currently offers both the strongest ranking signal and the best-calibrated probabilities."
    )

if float(final_choice_row["test_pr_auc"]) < 0.08:
    recommendation_lines.append(
        "Caution: discrimination remains weak, so this model should be framed as a portfolio-ranking aid rather than a high-confidence classifier."
    )

In [ ]:

print("FINAL RECOMMENDATION")
print("=" * 80)
for line in recommendation_lines:
    print("-", line)

final_model_artifact = {
    "model_name": champion_name,
    "selected_features": selected_features,
    "model": champion_model,
    "calibration_name": chosen_calibration,
    "calibrator": calibration_artifacts[chosen_calibration]["calibrator"],
    "threshold": calibration_artifacts[chosen_calibration]["threshold"],
    "preprocess_bundle_path": str(ARTIFACTS_DIR / "preprocess_bundle.joblib"),
    "reference_year": REFERENCE_YEAR,
}

joblib.dump(champion_model, ARTIFACTS_DIR / "champion_model.joblib")
joblib.dump(final_model_artifact, ARTIFACTS_DIR / "final_model.joblib")

best_params_payload = {}
tuned_champion_key = None
for name, bundle in tuned_candidates.items():
    if bundle["model"] is champion_model and "best_params" in bundle:
        tuned_champion_key = name
        best_params_payload = bundle["best_params"]
        break
if best_params_payload:
    with open(ARTIFACTS_DIR / "champion_best_params.json", "w", encoding="utf-8") as fp:
        json.dump(best_params_payload, fp, indent=2)

print("\nArtifacts written:")
print("-", ARTIFACTS_DIR / "preprocess_bundle.joblib")
print("-", ARTIFACTS_DIR / "champion_model.joblib")
print("-", ARTIFACTS_DIR / "final_model.joblib")
if best_params_payload:
    print("-", ARTIFACTS_DIR / "champion_best_params.json")

FINAL RECOMMENDATION
- Champion model before calibration: HistGradientBoosting
- Chosen calibration strategy: isotonic
- Best ranking profile on test: uncalibrated
- Best probability calibration on test (lowest Brier): isotonic
- Interpretation: this workflow likely produces one model that ranks risk best and another that emits more trustworthy probabilities.
- Caution: discrimination remains weak, so this model should be framed as a portfolio-ranking aid rather than a high-confidence classifier.

Artifacts written:
- artifacts/preprocess_bundle.joblib
- artifacts/champion_model.joblib
- artifacts/final_model.joblib
